# Person-in-WiFi-3D — micro-Doppler vs TSBA

**Runtime → Change runtime type → GPU (T4) trước khi chạy.**

Luồng mới: test → pilot loss cân bằng `(theta=40, rho=0.4, lambda_target=1)` → control `rho=0` → TSBA. **Không chạy sweep 25 ô nếu pilot chưa đi đúng target.**

Dữ liệu: giải nén vào `/content` (ổ local, nhanh hơn Drive nhiều).
Mất khi runtime restart → chạy lại cell 2. Checkpoint thì ghi ra Drive nên không mất.

## 1 · Tải shared folder bằng ID — không cần mount Drive

Folder phải bật **Anyone with the link → Viewer**. Code và dữ liệu chạy trong `/content`.
Checkpoint cũng ở `/content`, nên sau mỗi phase phải chạy cell **Backup** để tải về máy.

In [ ]:
import os, glob, subprocess, sys, torch
from pathlib import Path

FOLDER_ID = '1wh3lDLBgeZicPqGCgWTgLKJM3HKiQOQd'
FOLDER_URL = f'https://drive.google.com/drive/folders/{FOLDER_ID}'
BUNDLE = '/content/wifi_backdoor_bundle'
MARKER = f'{BUNDLE}/.download_complete'
if not os.path.exists(MARKER):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'gdown'], check=True)
    subprocess.run(['gdown', '--folder', FOLDER_URL, '-O', BUNDLE], check=True)
    Path(MARKER).touch()

cfg_hits = [p for p in glob.glob(f'{BUNDLE}/**/configs/sweep_colab.yaml', recursive=True)
            if '/configs/mmfi/' not in p]
if len(cfg_hits) != 1:
    raise RuntimeError(f'Không xác định được wbackdoor từ shared folder: {cfg_hits}')
WB = str(Path(cfg_hits[0]).parent.parent)
OUT = '/content/results_loss_v1/piw3d'
os.makedirs(OUT, exist_ok=True)

print('code      :', os.path.exists(f'{WB}/sweep.py'))
print('sweep cfg :', os.path.exists(f'{WB}/configs/sweep_colab.yaml'))
print('tsba  cfg :', os.path.exists(f'{WB}/configs/tsba_colab.yaml'))
print('GPU       :', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'CPU — bật GPU đi!')

## 2 · Dữ liệu về ổ local

Các asset được lấy từ shared folder vừa tải và giải nén vào ổ local.
Nếu cell báo thiếu file, kiểm tra folder link có chứa `piw3d_processed.zip`
và một file `data_bend.npy`.

In [ ]:
import os, glob, shutil

# trigger skeletons
if not os.path.exists('/content/psba_dataset/data_bend.npy'):
    hits = glob.glob(f'{BUNDLE}/**/data_bend.npy', recursive=True)
    if not hits:
        raise FileNotFoundError('Shared folder không có data_bend.npy')
    src = os.path.dirname(hits[0])
    shutil.copytree(src, '/content/psba_dataset', dirs_exist_ok=True)
    print('copied trigger skeletons')

# Person-in-WiFi-3D đã tiền xử lý (~8.5 GB)
if not os.path.exists('/content/piw3d/train_data/csi_ap'):
    zips = glob.glob(f'{BUNDLE}/**/piw3d_processed.zip', recursive=True)
    if not zips:
        raise FileNotFoundError('Shared folder không có piw3d_processed.zip')
    !mkdir -p /content/piw3d
    !unzip -q -o "{zips[0]}" -d /content/piw3d

for split in ['train_data', 'test_data']:
    d = f'/content/piw3d/{split}'
    n_ap = len(os.listdir(f'{d}/csi_ap')) if os.path.exists(f'{d}/csi_ap') else 0
    n_kp = len(os.listdir(f'{d}/keypoint')) if os.path.exists(f'{d}/keypoint') else 0
    print(f'{split}: csi_ap={n_ap}  keypoint={n_kp}')
print('trigger:', os.path.exists('/content/psba_dataset/data_bend.npy'))

## 3 · Kiểm tra trước khi tốn GPU

Chạy test suite và in cấu hình sẽ dùng. Nếu có gì đỏ ở đây, đừng chạy tiếp.

In [ ]:
%cd $WB
!python tests/test_checkpoint.py 2>&1 | tee -a $OUT/train.log
!python tests/test_distortion.py 2>&1 | tee -a $OUT/train.log
!python tests/test_regressions.py 2>&1 | tee -a $OUT/train.log

import yaml
for name in ['sweep_colab', 'tsba_colab']:
    c = yaml.safe_load(open(f'configs/{name}.yaml'))
    cells = len(c['theta']) * len(c['rho_grid'])
    print(f"\n{name}: trigger={c.get('trigger','micro_dropper')}  {cells} cell(s)"
          f"  theta={c['theta']}  rho={c['rho_grid']}")
    print(f"   pivot={c['pivot']}  eps={c['eps']}  dose={c['dose_mode']}"
          f"  select={c['poison_select']}  lr={c['lr']}  epochs={c['epochs']}")
    print(f"   lambda_target={c.get('lambda_target', 1.0)}"
          f"  lambda_nontarget={c.get('lambda_nontarget', 1.0)}")
    print(f"   dataset_root={c['dataset_root']}")

## 4 · PILOT loss cân bằng — 1 ô (θ=40, ρ=0.4, λt=1) ≈ 1.5–2 h

**Đừng bỏ qua bước này.** Sau khi xong, đọc bảng ở cell 4b:
Theo dõi log `clean`, `pois_t`, `pois_nt`. Bằng chứng targeted attack là
`target_residual_ratio < 1`, `target_progress > 0`, cosine và effect gain
dương — không phải chỉ riêng displacement hoặc Spearman.

In [ ]:
%cd $WB
!python sweep.py --config configs/sweep_colab.yaml \
    --theta 40 --rho 0.4 \
    --outdir $OUT/micro_lt1 2>&1 | tee -a $OUT/train.log

### 4b · Đọc kết quả pilot

In [ ]:
import json
r = json.load(open(f'{OUT}/micro_lt1/sweep_results.json'))[0]

print('PAYLOAD vs SAI SỐ NỀN')
print(f"  displacement        {r['displacement_mm']:8.1f} mm   (chi dịch được bao nhiêu)")
print(f"  clean_target_floor  {r['clean_target_floor_mm']:8.1f} mm   (model tự sai bao nhiêu)")
ratio = r['displacement_mm'] / max(r['clean_target_floor_mm'], 1e-9)
print(f"  tỉ lệ               {ratio:8.2f}      {'payload nổi' if ratio > 1 else 'PAYLOAD CHÌM dưới noise floor'}")

print('\nATTACK')
print(f"  asr                     {r['asr']:8.3f}")
print(f"  target_residual_ratio   {r['target_residual_ratio']:8.3f}   (<= 0.50 mới tính landed; 1.0 = model không phản ứng)")
print(f"  cosine_alignment       {r['cosine_alignment']:8.3f}   (hướng; gần +1 là đúng)")
print(f"  effect_gain            {r['effect_gain']:8.3f}   (biên độ; gần 1 là đủ)")
print(f"  target_progress        {r['target_progress']:8.3f}   (>0 mới đi về target)")
print(f"  frac_landed/preserved   {r['frac_landed']:.3f} / {r['frac_preserved']:.3f}")

print('\nINPUT SENSITIVITY / DOSE SHAPE  (không tự chứng minh backdoor)')
print(f"  spearman            {r['spearman']:8.3f}")
print(f"  ramp_minus_step     {r['ramp_minus_step']:8.3f}   (>0 = liên tục, không phải bật/tắt)")

print('\nCLEAN FIDELITY')
print(f"  clean_mpjpe         {r['clean_mpjpe_mm']:8.1f} mm")
print(f"  n_poison/n_total    {r['n_poison']}/{r['n_total']}")
print(f"  lambda_t/lambda_nt  {r['lambda_target']}/{r['lambda_nontarget']}")
direction_ok = (r['target_residual_ratio'] < 1 and r['target_progress'] > 0
                and r['cosine_alignment'] > 0 and r['effect_gain'] > 0)
print('\nQUYẾT ĐỊNH:', 'có tín hiệu targeted — chạy control' if direction_ok
      else 'CHƯA targeted — dừng trước TSBA/grid, gửi log để kiểm tra')

## 5 · Negative control bắt buộc — cùng trigger, ρ=0

Model này không thấy poison khi train. Nếu control cũng có displacement/Spearman cao
thì đó chỉ là input sensitivity. So sánh bằng residual, cosine, gain và progress.

In [ ]:
%cd $WB
!python sweep.py --config configs/sweep_colab.yaml \
    --theta 40 --rho 0 \
    --outdir $OUT/control_rho0 2>&1 | tee -a $OUT/train.log

import json
attack = json.load(open(f'{OUT}/micro_lt1/sweep_results.json'))[0]
control = json.load(open(f'{OUT}/control_rho0/sweep_results.json'))[0]
keys = ['clean_mpjpe_mm', 'asr', 'target_residual_ratio',
        'cosine_alignment', 'effect_gain', 'target_progress',
        'nontarget_ratio', 'displacement_mm', 'spearman']
print(f"{'metric':24s} {'rho=0':>12s} {'rho=0.4':>12s}")
for k in keys:
    print(f"{k:24s} {control[k]:12.4f} {attack[k]:12.4f}")

### 5b · Backup về máy — chạy lại sau mỗi phase

Không mount Drive nên runtime reset sẽ xoá checkpoint. Cell này ZIP toàn bộ
output hiện có và tải về máy; đợi tải xong rồi mới đóng runtime.

In [ ]:
from google.colab import files
import shutil, os
archive = shutil.make_archive('/content/piw3d_results_loss_v1', 'zip', OUT)
print('backup:', archive, f'{os.path.getsize(archive)/1e6:.1f} MB')
files.download(archive)

## 6 · TSBA baseline — 1 ô, cùng (θ=40, ρ=0.4)

Chậm hơn micro-Doppler vì mỗi batch còn cập nhật generator.
Theo dõi `tsba_loss` trong log: nếu nó **không giảm**, TSBA yếu là do
generator train hỏng, không phải kết luận về phương pháp.

In [ ]:
%cd $WB
!python sweep.py --config configs/tsba_colab.yaml \
    --outdir $OUT/tsba 2>&1 | tee -a $OUT/train.log

## 7 · Distortion đo được (cho bảng so sánh)

`eps` của hai trigger không cùng đơn vị nên không so trực tiếp được.
Đây là mức nhiễu thực tế trên tensor mà model nhận.
TSBA **bắt buộc** phải nạp generator đã train — công cụ sẽ từ chối nếu thiếu.

In [ ]:
%cd $WB
print('=== micro-Doppler ===')
!python -m eval.distortion --config configs/sweep_colab.yaml --n 128

print('\n=== TSBA (generator đã train) ===')
!python -m eval.distortion --config configs/tsba_colab.yaml --n 128 \
    --checkpoint $OUT/tsba/checkpoints/tsba_theta_40_rho_0p4/checkpoint.pt

## 8 · Sensitivity sweep 5×5 — chỉ chạy sau khi pilot + control ổn

25 ô × ~1.5 h ≈ **37 h**. Colab ngắt sau ~12 h, nên chạy **từng ρ một**
(5 ô ≈ 7.5 h mỗi đợt). Checkpoint resume-safe: chạy lại sẽ bỏ qua ô đã xong.

Bảng so sánh với TSBA phải lấy ô **(40, 0.4)** trong lưới này, không phải ô tốt nhất.

In [ ]:
%cd $WB
# đổi --rho cho mỗi đợt: 0.3 -> 0.35 -> 0.4 -> 0.45 -> 0.5
!python sweep.py --config configs/sweep_colab.yaml --rho 0.3 \
    --outdir $OUT/grid 2>&1 | tee -a $OUT/train.log

## 9 · Ablation C & D — mỗi cái 1 ô tại (θ=40, ρ=0.4)

`--outdir` riêng cho từng cái: `_cell_tag` chỉ gồm (θ, ρ, trigger) nên
chúng sẽ trùng tên thư mục nếu dùng chung outdir.

In [ ]:
%cd $WB
for name in ['ablation_dose_sqrt', 'ablation_dose_quad', 'ablation_select_uniform']:
    print(f'\n===== {name} =====')
    !python sweep.py --config configs/{name}.yaml --outdir $OUT/{name} 2>&1 | tee -a $OUT/train.log

## 10 · Tổng hợp kết quả

In [ ]:
import json, glob, os
rows = []
for f in sorted(glob.glob(f'{OUT}/**/sweep_results.json', recursive=True)):
    tag = os.path.basename(os.path.dirname(f))
    for r in json.load(open(f)):
        rows.append(dict(run=tag, **r))

if not rows:
    print('chưa có kết quả nào')
else:
    cols = ['run', 'theta_max_deg', 'rho', 'lambda_target', 'clean_mpjpe_mm',
            'asr', 'target_residual_ratio', 'cosine_alignment', 'effect_gain',
            'target_progress', 'nontarget_ratio', 'displacement_mm', 'spearman']
    print(''.join(f'{c[:13]:>15}' for c in cols))
    print('-' * (15 * len(cols)))
    for r in rows:
        print(''.join(f'{r.get(c, ""):>15.3f}' if isinstance(r.get(c), float)
                      else f'{str(r.get(c, ""))[:13]:>15}' for c in cols))
    try:
        import pandas as pd
        pd.DataFrame(rows).to_csv(f'{OUT}/all_results.csv', index=False)
        print(f'\nwrote {OUT}/all_results.csv')
    except ImportError:
        pass